# 두 카메라 교차 추적 v2

입구 정면 카메라에서 SCRFD·ArcFace로 얻은 신원을 입구 ByteTrack에 부착하고, 겹침 구역에서 OSNet-AIN ReID·시간·위치를 이용해 교실 카메라 ByteTrack으로 단방향 인계합니다. 교실 카메라에서는 얼굴 인식을 실행하지 않습니다.

집 테스트 기본값은 입구=`HOME_CAM_*` RTSP, 교실=`CLASSROOM_CAMERA_SOURCE=webcam`입니다. 학원에서는 두 역할의 source를 RTSP로 변경합니다.

## 1. OSNet-AIN 모델 준비

최초 한 번만 실행합니다. 공식 다중 도메인 가중치를 SHA-256 검증 후 Git 제외 경로에 내려받아 ONNX로 변환합니다. 이미 준비된 경우 검증만 수행합니다.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'deeplearning').is_dir() and (candidate / 'webapps').is_dir():
            return candidate
    raise RuntimeError('smart_office_monitoring 저장소 안에서 실행하세요.')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from deeplearning.training.prepare_person_reid import prepare as prepare_person_reid
prepare_person_reid()


OSNet-AIN ONNX 준비 완료: C:\smart_office_monitoring\deeplearning\.models\person_reid\osnet_ain_x1_0_msmt17.onnx


WindowsPath('C:/smart_office_monitoring/deeplearning/.models/person_reid/osnet_ain_x1_0_msmt17.onnx')

## 2. 두 카메라 실행과 최초 보정

보정 파일이 없으면 네 개의 창이 순서대로 열립니다.

1. 입구 카메라의 겹침 영역 폴리곤을 클릭하고 Enter
2. 교실 카메라의 같은 겹침 영역을 클릭하고 Enter
3. 입구 영상의 바닥 대응점 4개를 클릭하고 Enter
4. 교실 영상에서 같은 실제 지점 4개를 **같은 순서**로 클릭하고 Enter

`r`은 다시 찍기, `Esc`는 취소입니다. 실행 화면에서 `q`를 누르면 종료하고 진단 JSON을 저장합니다. 다시 보정하려면 `.env.face`에서 `CROSS_CAMERA_RECALIBRATE=true`로 바꾼 뒤 커널을 재시작합니다.

In [ ]:
from deeplearning.training.cross_camera_demo import run
run()


## 판독 방법

왼쪽 입구 화면의 `E<local> G<global> 이름`이 오른쪽 교실 화면에서 `C<local> G<global> 이름`으로 유지되어야 합니다. 오른쪽의 `unmapped`는 아직 겹침 구역에서 안전한 인계가 성립하지 않았다는 뜻입니다. 얼굴 후보 박스와 embedding은 화면이나 파일에 저장하지 않습니다.